## DATA MINING-ll PROJECT — Remaining 4 Models

**Advisor:** Dr. BIN LUO

### Team Members
- Anil Vallepu
- Param Venkat Vivek Kesireddy
- Vineetha Burugupalli

---

This notebook runs the **4 models that were skipped** in the 7-model notebook due to GPU/memory constraints:

| Model | Category | Issue in 7-model run | Fix applied here |
|---|---|---|---|
| TimeLLM | Large TS Model | Default `patch_size=24` crashes on EPA-Air (Feature Observability=0.38) | `patch_size=4` |
| CRU | Irregular (ODE) | ODE solver memory overhead | `batch_size=4` + `gc.collect()` |
| LatentODE | Irregular (ODE) | ODE solver memory overhead | `batch_size=4` + `gc.collect()` |
| NeuralFlow | Irregular (ODE) | ODE solver memory overhead | `batch_size=4` + `gc.collect()` |

**Paper target (Table 11 — EPA-Air):**
- TimeLLM: Uni=0.5835, Multi=0.5334
- CRU: Uni=0.7026, Multi=0.7982
- LatentODE: Uni=0.8025, Multi=0.7556
- NeuralFlow: Uni=0.7821, Multi=0.8202

Results are saved to `runs_4models.jsonl` and merged with `runs.jsonl` (from the 7-model notebook) at the end.

In [1]:
# ── Cell 1: GPU / RAM check ──────────────────────────────────────────────────
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader
!free -h | head -2
import torch
print(f"PyTorch: {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")

NVIDIA A100-SXM4-80GB, 81920 MiB, 0 MiB
               total        used        free      shared  buff/cache   available
Mem:           167Gi       967Mi       162Gi       2.0Mi       3.5Gi       164Gi
PyTorch: 2.10.0+cu128  |  CUDA: True


In [2]:
# ── Cell 2: Mount Google Drive + project directory setup ────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/STAT8240_EPA_Project'
for sub in ['results', 'embeddings', 'checkpoints', 'logs', 'kaggle']:
    os.makedirs(f'{PROJECT_DIR}/{sub}', exist_ok=True)
print(f'Project dir: {PROJECT_DIR}')
!ls {PROJECT_DIR}

Mounted at /content/drive
Project dir: /content/drive/MyDrive/STAT8240_EPA_Project
checkpoints  embeddings_BERT   kaggle  results
embeddings   embeddings_NOISE  logs


In [3]:
# ── Cell 3: Kaggle credentials (reuse from Drive) ───────────────────────────
import shutil

KAGGLE_DRIVE = f'{PROJECT_DIR}/kaggle/kaggle.json'
KAGGLE_HOME  = '/root/.kaggle/kaggle.json'
os.makedirs('/root/.kaggle', exist_ok=True)

if os.path.exists(KAGGLE_DRIVE):
    shutil.copy(KAGGLE_DRIVE, KAGGLE_HOME)
    print('✓ kaggle.json restored from Drive')
else:
    from google.colab import files
    print('Upload kaggle.json (one-time):')
    uploaded = files.upload()
    shutil.move('kaggle.json', KAGGLE_HOME)
    shutil.copy(KAGGLE_HOME, KAGGLE_DRIVE)
    print('✓ kaggle.json saved to Drive')

os.chmod(KAGGLE_HOME, 0o600)
!pip install -q kaggle

✓ kaggle.json restored from Drive


In [4]:
# ── Cell 4: Clone IMM-TSF repo ───────────────────────────────────────────────
%cd /content
!rm -rf Time-IMM IMM-TSF
!git clone -q https://github.com/blacksnail789521/Time-IMM.git
!git clone -q https://github.com/blacksnail789521/IMM-TSF.git
!ls

/content
drive  IMM-TSF	sample_data  Time-IMM


In [5]:
# ── Cell 5: Install dependencies ────────────────────────────────────────────
# Colab A100 has CUDA-enabled torch pre-installed.
# requirements.txt pins torch==2.7.0 (CPU build from PyPI) which overwrites it.
# Solution: filter out torch/torchvision/torchaudio lines before installing.

%cd /content/IMM-TSF

# Strip torch lines so pip doesn't downgrade to CPU build
!grep -iv "^torch" requirements.txt > /tmp/req_no_torch.txt
!pip install -q -r /tmp/req_no_torch.txt 2>&1 | tail -5

# ODE-based model extras (CRU, LatentODE, NeuralFlow)
!pip install -q reformer_pytorch==1.4.4 stribor==0.1.0 geotorch==0.3.0 2>&1 | tail -3

import torch
assert torch.cuda.is_available(), "CUDA not available — check Runtime > Change runtime type → GPU (A100)"
print(f'\n✓ PyTorch {torch.__version__} | CUDA {torch.version.cuda} | {torch.cuda.get_device_name(0)}')

/content/IMM-TSF
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.7/354.7 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 117.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.2.3 which is incompatible.

✓ PyTorch 2.10.0+cu128 | CUDA 12.8 | NVIDIA A100-SXM4-80GB


In [6]:
# ── Cell 6: Global matplotlib style (identical to 7-model notebook) ─────────
import matplotlib.pyplot as plt
import matplotlib as mpl

PALETTE = {
    'primary':   '#065A82',
    'secondary': '#E8A33D',
    'accent':    '#C84B31',
    'neutral':   '#2C3539',
    'muted':     '#B8C5D1',
    'success':   '#4A7C59',
}

mpl.rcParams.update({
    'figure.dpi': 110,
    'savefig.dpi': 200,
    'savefig.bbox': 'tight',
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.edgecolor': PALETTE['neutral'],
    'axes.labelcolor': PALETTE['neutral'],
    'xtick.color': PALETTE['neutral'],
    'ytick.color': PALETTE['neutral'],
    'axes.grid': True,
    'grid.color': PALETTE['muted'],
    'grid.alpha': 0.4,
    'grid.linewidth': 0.6,
    'legend.frameon': False,
    'legend.fontsize': 10,
})
print('✓ Matplotlib style loaded')

✓ Matplotlib style loaded


In [7]:
# ── Cell 7: Load / verify EPA-Air dataset ───────────────────────────────────
# Data was already downloaded + cached in the 7-model notebook.
# This cell re-downloads only if the data directory is missing.
import pandas as pd
from pathlib import Path

EPA_PATH = '/content/IMM-TSF/data/EPA-Air'

if os.path.exists(EPA_PATH):
    print(f'✓ Data already present at {EPA_PATH} — skipping download')
else:
    !mkdir -p /content/IMM-TSF/data
    %cd /content/IMM-TSF/data
    !kaggle datasets download -d blacksnail789521/time-imm --unzip 2>&1 | tail -5
    %cd /content
    print('✓ Download complete')

epa_root    = Path('/content/IMM-TSF/data/EPA-Air/processed')
counties    = sorted([p.name for p in epa_root.iterdir() if p.is_dir()])
feature_cols = ['temp', 'pm2_5', 'aqi', 'ozone']

# Replicate Table 1 stats (paper: 8 entities, 4 features, 6587 timestamps, 49552 obs)
print(f'\n{"County":<25} {"Rows":>7} {"Non-null":>10} {"Missing":>10} {"Texts":>6}')
print('-' * 62)
total_rows = total_nonnull = total_text = 0
for c in counties:
    ts  = pd.read_csv(epa_root / c / 'time_series.csv')
    txt_path = epa_root / c / 'text.csv'
    txt = pd.read_csv(txt_path) if txt_path.exists() else pd.DataFrame()
    rows   = len(ts)
    nonnull = int(ts[feature_cols].notna().sum().sum())
    miss    = ts[feature_cols].isna().mean().mean() * 100
    print(f'{c:<25} {rows:>7} {nonnull:>10} {miss:>9.1f}% {len(txt):>6}')
    total_rows += rows; total_nonnull += nonnull; total_text += len(txt)
print('-' * 62)
print(f'{"TOTAL":<25} {total_rows:>7} {total_nonnull:>10} {"":>10} {total_text:>6}')
print()
print('Paper Table 1 reference (EPA-Air):')
print(f'  Entities: 8        (yours: {len(counties)})')
print(f'  Features: 4        (yours: {len(feature_cols)})')
print(f'  Timestamps: 6,587  (yours: {total_rows})')
print(f'  Observations: 49,552  (yours: {total_nonnull})')

/content/IMM-TSF/data
Dataset URL: https://www.kaggle.com/datasets/blacksnail789521/time-imm
License(s): Attribution 4.0 International (CC BY 4.0)
100%|██████████| 15.4M/15.4M [00:01<00:00, 15.0MB/s]

/content
✓ Download complete

County                       Rows   Non-null    Missing  Texts
--------------------------------------------------------------
Bexar                        4369       5123      70.7%     17
Dallas                       4369       5123      70.7%    209
Denver                       5957       6998      70.6%    213
Hillsborough                 4369       5123      70.7%     73
Los_Angeles                  5851       6860      70.7%    222
Maricopa                     6567       7702      70.7%     99
Philadelphia                 4337       5083      70.7%    203
Richmond                     6487       7540      70.9%    208
--------------------------------------------------------------
TOTAL                       42306      49552              1244

Paper Table 

In [8]:
# ── Cell 8: Restore GPT-2 embeddings from Drive ──────────────────────────────
# These were computed in the 7-model notebook and cached to Drive.
# Required for multimodal runs of all 4 models.
import torch

EMB_CACHE_DIR = f'{PROJECT_DIR}/embeddings/EPA-Air'
EPA_PROCESSED = '/content/IMM-TSF/data/EPA-Air/processed'

cached_pts = list(Path(EMB_CACHE_DIR).rglob('*.pt')) if os.path.exists(EMB_CACHE_DIR) else []

if cached_pts:
    print(f'✓ Found {len(cached_pts)} cached GPT-2 embedding files — restoring')
    for pt in cached_pts:
        county = pt.parent.name
        dest_dir = Path(EPA_PROCESSED) / county
        dest_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy(pt, dest_dir / pt.name)
    print('✓ Restored to /content/IMM-TSF/data/EPA-Air/processed')
else:
    print('⚠ No cached embeddings found!')
    print(f'  Expected: {EMB_CACHE_DIR}')
    print('  → Run the 7-model notebook first to compute + cache GPT-2 embeddings.')
    print('  → Unimodal runs will still work; multimodal runs will fail without embeddings.')

print('\nEmbedding verification per county:')
for county_dir in sorted(Path(EPA_PROCESSED).iterdir()):
    if county_dir.is_dir():
        pts = list(county_dir.glob('*.pt'))
        status = '✓' if pts else '✗'
        print(f'  {status} {county_dir.name}: {len(pts)} .pt files')

✓ Found 8 cached GPT-2 embedding files — restoring
✓ Restored to /content/IMM-TSF/data/EPA-Air/processed

Embedding verification per county:
  ✓ Bexar: 1 .pt files
  ✓ Dallas: 1 .pt files
  ✓ Denver: 1 .pt files
  ✓ Hillsborough: 1 .pt files
  ✓ Los_Angeles: 1 .pt files
  ✓ Maricopa: 1 .pt files
  ✓ Philadelphia: 1 .pt files
  ✓ Richmond: 1 .pt files


In [9]:
# ── Cell 9: Inspect available model arguments ────────────────────────────────
# Check what patch/token args main.py exposes — needed to fix TimeLLM and ODE models
print('=== Patch / token args ===')
!cd /content/IMM-TSF && python main.py --help 2>&1 | grep -iE "(patch|token|llm_model|input_len|pred_len)" | head -20
print()
print('=== ODE-specific args ===')
!cd /content/IMM-TSF && python main.py --help 2>&1 | grep -iE "(ode|latent|hidden|gru|flow|cru)" | head -20

=== Patch / token args ===
               [--stride STRIDE] [-ps PATCH_SIZE] [--npatch NPATCH]
               [--patch_stride PATCH_STRIDE] [--model MODEL]
               [--domain_des DOMAIN_DES] [--input_token_len INPUT_TOKEN_LEN]
               [--output_token_len OUTPUT_TOKEN_LEN]
               [--llm_model_timellm LLM_MODEL_TIMELLM]
               [--llm_model_fusion LLM_MODEL_FUSION]
  --stride STRIDE       Stride between consecutive patches
  -ps PATCH_SIZE, --patch_size PATCH_SIZE
                        Size of each temporal patch
  --npatch NPATCH       Number of patches (default: history/patch_size)
  --patch_stride PATCH_STRIDE
                        Stride between patches (defaults to patch_size)
                        number of attention patching levels
  --input_token_len INPUT_TOKEN_LEN
                        input token length
  --output_token_len OUTPUT_TOKEN_LEN
                        output token length
  --llm_model_timellm LLM_MODEL_TIMELLM
  --llm_model_fusi

In [10]:
# ── Cell 10: Runner setup ────────────────────────────────────────────────────
import subprocess, re, json, time, gc
from datetime import datetime

RESULTS_FILE = f'{PROJECT_DIR}/results/runs_4models.jsonl'
LOGS_DIR     = f'{PROJECT_DIR}/logs'
Path(LOGS_DIR).mkdir(parents=True, exist_ok=True)

DEFAULT_CONFIG = {
    'dataset':      'EPA-Air',
    'data_root':    '/content/IMM-TSF/data',
    'history':      7,
    'pred_window':  7,
    'stride':       7,
    'time_unit':    'days',
    'split_method': 'sample',
    'seed':         42,
    'gpu':          0,
    'batch_size':   8,
    'lr':           '1e-3',
    'epoch':        50,
    'patience':     10,
}

DEFAULT_MULTI = {
    'TTF_module':       'TTF_RecAvg',
    'MMF_module':       'MMF_GR_Add',
    'llm_model_fusion': 'GPT2',
}


def parse_output(stdout):
    out = {'mse': None, 'mae': None}
    mse_matches = re.findall(r'^mse:\s*([\d.]+)', stdout, flags=re.MULTILINE)
    mae_matches = re.findall(r'^mae:\s*([\d.]+)', stdout, flags=re.MULTILINE)
    if mse_matches: out['mse'] = float(mse_matches[-1])
    if mae_matches: out['mae'] = float(mae_matches[-1])
    return out


def run_model(model, mode='uni', notes='', show_tail=True, proc_timeout=3600, **overrides):
    cfg = {**DEFAULT_CONFIG, 'model': model, **overrides}
    cmd = ['python', 'main.py']
    for k, v in cfg.items():
        cmd += [f'--{k}', str(v)]
    if mode == 'multi':
        cmd += ['--enable_text', '--use_text_embeddings']
        for k, v in DEFAULT_MULTI.items():
            cmd += [f'--{k}', str(v)]

    print(f'▶ {model} ({mode})')

    t0   = time.time()
    proc = subprocess.run(
        cmd, cwd='/content/IMM-TSF',
        capture_output=True, text=True, timeout=proc_timeout
    )
    elapsed = time.time() - t0

    log_name = f'{model}_{mode}_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'
    (Path(LOGS_DIR) / log_name).write_text(
        f'CMD: {" ".join(cmd)}\n\n=== STDOUT ===\n{proc.stdout}'
        f'\n\n=== STDERR ===\n{proc.stderr}'
    )

    if proc.returncode != 0:
        print(f'   ✗ FAILED ({elapsed:.0f}s, exit={proc.returncode}) — see {log_name}')
        if show_tail:
            print('\n'.join('   ' + l for l in proc.stderr.splitlines()[-25:]))
        return None

    res = parse_output(proc.stdout)
    if res['mse'] is not None:
        print(f'   ✓ MSE={res["mse"]:.4f}  MAE={res["mae"]:.4f}  time={elapsed:.0f}s')
    else:
        print(f'   ⚠ Parse failed — see {log_name}')
        if show_tail:
            print('\n'.join('   ' + l for l in proc.stdout.splitlines()[-15:]))

    entry = {
        'timestamp':  datetime.now().isoformat(),
        'model':      model,
        'mode':       mode,
        'mse':        res['mse'],
        'mae':        res['mae'],
        'elapsed_s':  round(elapsed, 1),
        'seed':       cfg['seed'],
        'dataset':    cfg['dataset'],
        'log_file':   log_name,
        'notes':      notes,
    }
    if mode == 'multi':
        entry.update(DEFAULT_MULTI)

    with open(RESULTS_FILE, 'a') as f:
        f.write(json.dumps(entry) + '\n')

    return entry


print('✓ Runner ready.  Results → runs_4models.jsonl')

✓ Runner ready.  Results → runs_4models.jsonl


# ── Cell 11b: Restore TimeLLM.py to original (undo all prior patches) ────────
# Previous patch attempts used -1 in view/reshape which made things WORSE
# because they exposed a deeper bug: d_ff=2048 > GPT-2 d_llm=768.
# The correct fix is to use paper config (d_ff=128, d_model=32) — no source edit needed.

import subprocess, os

timellm_path = '/content/IMM-TSF/models/TimeLLM.py'
bak_path     = timellm_path + '.bak'

# Try git restore first (cleanest); fall back to .bak if git fails
r = subprocess.run(
    ['git', 'checkout', 'models/TimeLLM.py'],
    cwd='/content/IMM-TSF', capture_output=True, text=True
)
if r.returncode == 0:
    print('✓ TimeLLM.py restored via git checkout')
elif os.path.exists(bak_path):
    import shutil
    shutil.copy(bak_path, timellm_path)
    print('✓ TimeLLM.py restored from .bak')
else:
    print('⚠ Could not restore — re-clone IMM-TSF if needed')

# Verify: the -1 patches should NOT be present
with open(timellm_path) as f:
    src = f.read()
patched_count = src.count('view(B, N, -1,') + src.count('view(B, -1,') + src.count('reshape(B * n_vars, self.d_ff, -1)')
if patched_count == 0:
    print('✓ File is clean (no -1 patches present)')
else:
    print(f'⚠ {patched_count} -1 patches still present — re-clone IMM-TSF')

In [11]:
# ── Cell 12: TimeLLM — paper config, no source patches needed ────────────────
#
# ROOT CAUSE OF ALL PRIOR ERRORS:
#   configs.d_ff = 2048  (default transformer d_ff)
#   GPT-2 d_llm  = 768   (GPT-2 hidden size)
#
#   The line:  dec = llama_out[:, -total_ts:, :self.d_ff]
#   only extracts 768 features (not 2048) because GPT-2 outputs 768-wide tensors.
#   This causes dec.shape = [8, 96, 768] instead of [8, 96, 2048], which makes
#   all subsequent view/reshape calls produce wrong shapes.
#
# CORRECT FIX (paper Appendix K.1 values):
#   d_ff=128, d_model=32 — both < d_llm=768 so extraction works correctly.
#   No source-level patch to TimeLLM.py is needed.
#
# TENSOR FLOW (verified):
#   patch_nums = (input_len=168 - input_token_len=16) // stride=7 + 2 = 23
#   dec = llama_out[:, -92:, :128]           → [8, 92, 128]   (92 = 23*4 vars)
#   dec.view(B, patch_nums, n_vars, d_ff)    → [8, 23, 4, 128] (8*23*4*128=94208=8*92*128 ✓)
#   dec.permute(0,2,3,1)                     → [8, 4, 128, 23]
#   dec.reshape(B*n_vars, d_ff, patch_nums)  → [32, 128, 23]              ✓
#   FlattenHead: Linear(128*23=2944, 168)    → [32, 168]                  ✓

import torch, gc

print('=' * 65)
print('TimeLLM — paper config: d_ff=128, d_model=32, input_token_len=16')
print('No TimeLLM.py source patches required.')
print('=' * 65)

torch.cuda.empty_cache(); gc.collect()

run_model('TimeLLM', 'uni',   notes='Phase 4 reproduction',
          input_token_len=16, d_model=32, d_ff=128)
torch.cuda.empty_cache(); gc.collect()

run_model('TimeLLM', 'multi', notes='Phase 4 reproduction',
          input_token_len=16, d_model=32, d_ff=128)
torch.cuda.empty_cache(); gc.collect()

print('\nPaper Table 11 targets  →  Uni MSE: 0.5835   Multi MSE: 0.5334')

TimeLLM — paper config: d_ff=128, d_model=32, input_token_len=16
No TimeLLM.py source patches required.
▶ TimeLLM (uni)
   ✓ MSE=0.6138  MAE=0.5777  time=60s
▶ TimeLLM (multi)
   ✓ MSE=0.5363  MAE=0.5370  time=63s

Paper Table 11 targets  →  Uni MSE: 0.5835   Multi MSE: 0.5334


## CRU (Continuous Recurrent Unit)

**Paper config (Appendix K.1):** Latent state and hidden units = 32, square/exponential activations, gravity gates enabled.

**Why it failed previously:** ODE-based numerical integration at every forward pass creates extreme memory overhead when EPA-Air's dense irregular timestamps (Mean IOI = 1.02 hours) force very fine-grained ODE solving.

**Fix:** `batch_size=4` (halved from 8) + explicit `torch.cuda.empty_cache()` + `gc.collect()` between runs.

In [ ]:
# ── CRU / LatentODE / NeuralFlow — paste into a new cell, run directly ──
import torch, gc, subprocess, re, json, time
from datetime import datetime

torch.cuda.empty_cache(); gc.collect()

def _run_ode(model, mode, epoch=10, patience=3):
    cmd = [
        'python', 'main.py',
        '--dataset', 'EPA-Air', '--data_root', '/content/IMM-TSF/data',
        '--history', '7', '--pred_window', '7', '--stride', '7',
        '--time_unit', 'days', '--split_method', 'sample',
        '--seed', '42', '--gpu', '0',
        '--model', model, '--batch_size', '4',
        '--lr', '1e-3', '--epoch', str(epoch), '--patience', str(patience),
    ]
    if mode == 'multi':
        cmd += ['--enable_text', '--use_text_embeddings',
                '--TTF_module', 'TTF_RecAvg',
                '--MMF_module', 'MMF_GR_Add',
                '--llm_model_fusion', 'GPT2']
    print(f'▶ {model} ({mode})  epoch={epoch}')
    t0 = time.time()
    proc = subprocess.run(cmd, cwd='/content/IMM-TSF',
                          capture_output=True, text=True, timeout=7200)
    elapsed = time.time() - t0
    if proc.returncode != 0:
        print(f'  ✗ FAILED ({elapsed:.0f}s)\n' + proc.stderr[-1500:])
        return
    mse = re.findall(r'^mse:\s*([\d.]+)', proc.stdout, re.MULTILINE)
    mae = re.findall(r'^mae:\s*([\d.]+)', proc.stdout, re.MULTILINE)
    mv, av = (float(mse[-1]) if mse else None), (float(mae[-1]) if mae else None)
    print(f'  ✓ MSE={mv:.4f}  MAE={av:.4f}  time={elapsed:.0f}s')
    entry = {'timestamp': datetime.now().isoformat(), 'model': model, 'mode': mode,
             'mse': mv, 'mae': av, 'elapsed_s': round(elapsed,1),
             'seed': 42, 'dataset': 'EPA-Air', 'notes': f'Phase4 epoch={epoch}'}
    if mode == 'multi':
        entry.update({'TTF_module':'TTF_RecAvg','MMF_module':'MMF_GR_Add','llm_model_fusion':'GPT2'})
    with open(RESULTS_FILE, 'a') as f:
        f.write(json.dumps(entry) + '\n')
    torch.cuda.empty_cache(); gc.collect()

for model in ['CRU', 'LatentODE', 'NeuralFlow']:
    _run_ode(model, 'uni')
    _run_ode(model, 'multi')

print('\nAll ODE models done. Check RESULTS_FILE for entries.')

# This runs all three ODE models back-to-back with epoch=10, timeout=7200.
# RESULTS_FILE must already be defined from Cell 10 — if not, add RESULTS_FILE =
#  f'{PROJECT_DIR}/results/runs_4models.jsonl' at the top.

▶ CRU (uni)  epoch=10
  ✓ MSE=0.8087  MAE=0.6715  time=1468s
▶ CRU (multi)  epoch=10
  ✓ MSE=0.9212  MAE=0.7300  time=1502s
▶ LatentODE (uni)  epoch=10
  ✓ MSE=0.8401  MAE=0.6857  time=4320s
▶ LatentODE (multi)  epoch=10
  ✓ MSE=0.8940  MAE=0.7107  time=5310s
▶ NeuralFlow (uni)  epoch=10


In [12]:
# ── Cell 12: CRU (ODE-based irregular TSF model) ─────────────────────────────
# Paper Appendix K.1:
#   Latent state + hidden = 32
#   square/exponential activations, gravity gates enabled
# Fix: batch_size=4 to reduce ODE solver peak memory

print('=' * 65)
print('CRU — Continuous Recurrent Unit (ODE-based)')
print('batch_size=4, explicit GPU memory flush between runs')
print('=' * 65)

torch.cuda.empty_cache(); gc.collect()

run_model('CRU', 'uni',   notes='Phase 4 reproduction', batch_size=4)
torch.cuda.empty_cache(); gc.collect()

run_model('CRU', 'multi', notes='Phase 4 reproduction', batch_size=4)
torch.cuda.empty_cache(); gc.collect()

print()
print('Paper Table 11 targets  →  Uni: 0.7026   Multi: 0.7982')

CRU — Continuous Recurrent Unit (ODE-based)
batch_size=4, explicit GPU memory flush between runs
▶ CRU (uni)


TimeoutExpired: Command '['python', 'main.py', '--dataset', 'EPA-Air', '--data_root', '/content/IMM-TSF/data', '--history', '7', '--pred_window', '7', '--stride', '7', '--time_unit', 'days', '--split_method', 'sample', '--seed', '42', '--gpu', '0', '--batch_size', '4', '--lr', '1e-3', '--epoch', '50', '--patience', '10', '--model', 'CRU']' timed out after 3600 seconds

# ── Cell 15: CRU ─────────────────────────────────────────────────────────────
# CRU timed out at 1 hr with epoch=50. Each epoch ~72s on A100 for EPA-Air.
# epoch=10 → ~12 min total; well within 2-hr budget.
import torch, gc

print('=' * 65)
print('CRU — batch_size=4, epoch=10, patience=3, proc_timeout=7200s')
print('=' * 65)

torch.cuda.empty_cache(); gc.collect()
run_model('CRU', 'uni',   notes='Phase 4 reproduction',
          batch_size=4, epoch=10, patience=3, proc_timeout=7200)
torch.cuda.empty_cache(); gc.collect()
run_model('CRU', 'multi', notes='Phase 4 reproduction',
          batch_size=4, epoch=10, patience=3, proc_timeout=7200)
torch.cuda.empty_cache(); gc.collect()

print('\nPaper Table 11 targets  →  Uni: 0.7026   Multi: 0.7982')

In [ ]:
# ── Cell 13: LatentODE ────────────────────────────────────────────────────────
# Paper Appendix K.1:
#   Recognition GRU hidden=32, 1 RNN + 1 generator layer, ODE units=32

print('=' * 65)
print('LatentODE (ODE-based irregular TSF model)')
print('Recognition GRU hidden=32, ODE units=32, batch_size=4')
print('=' * 65)

torch.cuda.empty_cache(); gc.collect()

run_model('LatentODE', 'uni',   notes='Phase 4 reproduction', batch_size=4)
torch.cuda.empty_cache(); gc.collect()

run_model('LatentODE', 'multi', notes='Phase 4 reproduction', batch_size=4)
torch.cuda.empty_cache(); gc.collect()

print()
print('Paper Table 11 targets  →  Uni: 0.8025   Multi: 0.7556')

# ── Cell 17: LatentODE ───────────────────────────────────────────────────────
import torch, gc

print('=' * 65)
print('LatentODE — batch_size=4, epoch=10, patience=3, proc_timeout=7200s')
print('=' * 65)

torch.cuda.empty_cache(); gc.collect()
run_model('LatentODE', 'uni',   notes='Phase 4 reproduction',
          batch_size=4, epoch=10, patience=3, proc_timeout=7200)
torch.cuda.empty_cache(); gc.collect()
run_model('LatentODE', 'multi', notes='Phase 4 reproduction',
          batch_size=4, epoch=10, patience=3, proc_timeout=7200)
torch.cuda.empty_cache(); gc.collect()

print('\nPaper Table 11 targets  →  Uni: 0.8025   Multi: 0.7556')

In [ ]:
# ── Cell 14: NeuralFlow ──────────────────────────────────────────────────────
# Paper Appendix K.1:
#   latent_dim=20, 3-layer hidden (dim=32), GRU units=32
#   coupling flow with 2 layers, time network: TimeLinear

print('=' * 65)
print('NeuralFlow (ODE/Flow-based irregular TSF model)')
print('latent_dim=20, hidden=32, GRU=32, coupling_flow=2, batch_size=4')
print('=' * 65)

torch.cuda.empty_cache(); gc.collect()

run_model('NeuralFlow', 'uni',   notes='Phase 4 reproduction', batch_size=4)
torch.cuda.empty_cache(); gc.collect()

run_model('NeuralFlow', 'multi', notes='Phase 4 reproduction', batch_size=4)
torch.cuda.empty_cache(); gc.collect()

print()
print('Paper Table 11 targets  →  Uni: 0.7821   Multi: 0.8202')

# ── Cell 19: NeuralFlow ──────────────────────────────────────────────────────
import torch, gc

print('=' * 65)
print('NeuralFlow — batch_size=4, epoch=10, patience=3, proc_timeout=7200s')
print('=' * 65)

torch.cuda.empty_cache(); gc.collect()
run_model('NeuralFlow', 'uni',   notes='Phase 4 reproduction',
          batch_size=4, epoch=10, patience=3, proc_timeout=7200)
torch.cuda.empty_cache(); gc.collect()
run_model('NeuralFlow', 'multi', notes='Phase 4 reproduction',
          batch_size=4, epoch=10, patience=3, proc_timeout=7200)
torch.cuda.empty_cache(); gc.collect()

print('\nPaper Table 11 targets  →  Uni: 0.7821   Multi: 0.8202')

In [ ]:
# ── Cell 15: Results table — 4 new models ───────────────────────────────────
import json, pandas as pd

MODELS_4 = ['TimeLLM', 'CRU', 'LatentODE', 'NeuralFlow']

# Paper Table 11 reference values for EPA-Air
paper_ref_4 = {
    'TimeLLM':    {'paper_uni': 0.5835, 'paper_multi': 0.5334},
    'CRU':        {'paper_uni': 0.7026, 'paper_multi': 0.7982},
    'LatentODE':  {'paper_uni': 0.8025, 'paper_multi': 0.7556},
    'NeuralFlow': {'paper_uni': 0.7821, 'paper_multi': 0.8202},
}

rows = []
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE) as f:
        for line in f:
            e = json.loads(line)
            if e.get('mse') is not None and e['model'] in MODELS_4:
                rows.append(e)

df4 = pd.DataFrame(rows)
if df4.empty:
    print('⚠ No results yet — run models above first.')
else:
    df4 = df4.sort_values('timestamp').drop_duplicates(subset=['model', 'mode'], keep='last')

    # Build pivot table
    piv = df4.pivot(index='model', columns='mode', values=['mse', 'mae'])
    piv.columns = [f'{m}_{md}' for m, md in piv.columns]
    piv = piv.reset_index()

    ref_df = pd.DataFrame(paper_ref_4).T.reset_index().rename(columns={'index': 'model'})
    piv = piv.merge(ref_df, on='model', how='left')

    piv['mse_change_pct'] = ((piv['mse_multi'] - piv['mse_uni']) / piv['mse_uni'] * 100).round(1)
    piv['text_helps']     = piv['mse_change_pct'].apply(lambda x: '✓' if pd.notna(x) and x < 0 else '✗')

    print('=' * 95)
    print('4-Model Reproduction Results vs Paper (Table 11 — EPA-Air)')
    print('=' * 95)
    print(f'{"Model":<12} {"Uni (ours)":>12} {"Uni (paper)":>12} '
          f'{"Multi (ours)":>13} {"Multi (paper)":>14} {"Δ%":>7} {"Text?":>6}')
    print('-' * 95)
    for _, r in piv.iterrows():
        u  = f'{r["mse_uni"]:.4f}'   if pd.notna(r.get('mse_uni'))   else '  N/A  '
        m  = f'{r["mse_multi"]:.4f}' if pd.notna(r.get('mse_multi')) else '  N/A  '
        dlt = f'{r["mse_change_pct"]:+.1f}%' if pd.notna(r.get('mse_change_pct')) else '  N/A '
        print(f'{r["model"]:<12} {u:>12} {r["paper_uni"]:>12.4f} '
              f'{m:>13} {r["paper_multi"]:>14.4f} {dlt:>7} {r["text_helps"]:>6}')
    print('=' * 95)
    print(f'\nText helps: {(piv["text_helps"]=="✓").sum()}/4   '
          f'Text hurts: {(piv["text_helps"]=="✗").sum()}/4')

In [ ]:
# ── Cell 16: Bar chart — 4 models ────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

if 'piv' not in dir() or piv.empty:
    print('⚠ No data to plot — run Cell 15 first.')
else:
    models  = piv['model'].tolist()
    uni_v   = piv['mse_uni'].fillna(0).tolist()
    multi_v = piv['mse_multi'].fillna(0).tolist()
    changes = piv['mse_change_pct'].fillna(0).tolist()
    x = np.arange(len(models))
    w = 0.35

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - w/2, uni_v,   w, color=PALETTE['primary'],   label='Unimodal',           zorder=3)
    ax.bar(x + w/2, multi_v, w, color=PALETTE['secondary'], label='Multimodal (+ Text)', zorder=3)

    for i, (u, m, ch) in enumerate(zip(uni_v, multi_v, changes)):
        color = PALETTE['success'] if ch < 0 else PALETTE['accent']
        ax.text(i, max(u, m) + 0.015, f'{ch:+.1f}%',
                ha='center', va='bottom', fontsize=10, fontweight='bold', color=color)

    ax.set_xticks(x)
    ax.set_xticklabels(models, fontsize=11)
    ax.set_ylabel('MSE (lower is better)', fontsize=11)
    ax.set_title('EPA-Air: 4 Remaining Models — Does Text Help?',
                 fontsize=13, fontweight='bold', color=PALETTE['neutral'], pad=12)
    ax.set_ylim(0, max(max(uni_v), max(multi_v)) * 1.15)
    ax.legend(fontsize=10)

    plt.tight_layout()
    out = f'{PROJECT_DIR}/results/fig_4models_uni_vs_multi.png'
    plt.savefig(out, dpi=200, bbox_inches='tight', facecolor='white')
    print(f'✓ Saved to {out}')
    plt.show()

---
## Combined Results: All 11 Models

Merges results from:
- `runs.jsonl` — 7 regular/large/irregular models from the 7-model notebook
- `runs_4models.jsonl` — TimeLLM, CRU, LatentODE, NeuralFlow from this notebook

In [ ]:
# ── Cell 17: Load + merge results from both notebooks ────────────────────────
import json, pandas as pd, os

RESULTS_FILE_7  = f'{PROJECT_DIR}/results/runs.jsonl'
RESULTS_FILE_4  = f'{PROJECT_DIR}/results/runs_4models.jsonl'

ALL_MODELS = [
    'DLinear', 'Informer', 'PatchTST', 'TimesNet', 'TimeMixer',  # Regular
    'TimeLLM', 'TTM',                                              # Large TS
    'CRU', 'LatentODE', 'NeuralFlow',                             # Irregular ODE
    'tPatchGNN',                                                   # Irregular GNN
]

# Paper Table 11 — all 11 models on EPA-Air
paper_ref_all = {
    'Informer':   {'paper_uni': 0.6301, 'paper_multi': 0.5812},
    'DLinear':    {'paper_uni': 0.5361, 'paper_multi': 0.5223},
    'PatchTST':   {'paper_uni': 0.6196, 'paper_multi': 0.6204},
    'TimesNet':   {'paper_uni': 0.5599, 'paper_multi': 0.5892},
    'TimeMixer':  {'paper_uni': 0.6086, 'paper_multi': 0.5641},
    'TimeLLM':    {'paper_uni': 0.5835, 'paper_multi': 0.5334},
    'TTM':        {'paper_uni': 0.6002, 'paper_multi': 0.6218},
    'CRU':        {'paper_uni': 0.7026, 'paper_multi': 0.7982},
    'LatentODE':  {'paper_uni': 0.8025, 'paper_multi': 0.7556},
    'NeuralFlow': {'paper_uni': 0.7821, 'paper_multi': 0.8202},
    'tPatchGNN':  {'paper_uni': 0.6258, 'paper_multi': 0.5840},
}

all_rows = []
for fpath in [RESULTS_FILE_7, RESULTS_FILE_4]:
    if not os.path.exists(fpath):
        print(f'⚠ File not found: {fpath}')
        continue
    with open(fpath) as f:
        for line in f:
            try:
                e = json.loads(line)
                if e.get('mse') is not None and e['model'] in ALL_MODELS:
                    all_rows.append(e)
            except Exception:
                pass

df_all = pd.DataFrame(all_rows)
if df_all.empty:
    print('⚠ No results found in either file.')
else:
    # Keep latest run per (model, mode) pair
    df_all = df_all.sort_values('timestamp').drop_duplicates(
        subset=['model', 'mode'], keep='last'
    )

    piv_all = df_all.pivot(index='model', columns='mode', values=['mse', 'mae'])
    piv_all.columns = [f'{m}_{md}' for m, md in piv_all.columns]
    piv_all = piv_all.reset_index()

    ref_all = pd.DataFrame(paper_ref_all).T.reset_index().rename(columns={'index': 'model'})
    piv_all = piv_all.merge(ref_all, on='model', how='outer')  # outer keeps missing models

    piv_all['mse_change_pct'] = (
        (piv_all['mse_multi'] - piv_all['mse_uni']) / piv_all['mse_uni'] * 100
    ).round(1)
    piv_all['text_helps'] = piv_all['mse_change_pct'].apply(
        lambda x: '✓' if pd.notna(x) and x < 0 else ('✗' if pd.notna(x) else '?')
    )

    # Sort by paper_uni for display
    piv_all = piv_all.sort_values('paper_uni').reset_index(drop=True)

    print('=' * 100)
    print('COMBINED: All 11 Models on EPA-Air vs Paper (Table 11)')
    print('=' * 100)
    print(f'{"Model":<12} {"Uni(ours)":>10} {"Uni(paper)":>11} '
          f'{"Multi(ours)":>12} {"Multi(paper)":>13} {"Δ%":>7} {"Text?":>6}')
    print('-' * 100)
    for _, r in piv_all.iterrows():
        u   = f'{r["mse_uni"]:.4f}'   if pd.notna(r.get('mse_uni'))   else '  N/A '
        m   = f'{r["mse_multi"]:.4f}' if pd.notna(r.get('mse_multi')) else '  N/A '
        dlt = f'{r["mse_change_pct"]:+.1f}%' if pd.notna(r.get('mse_change_pct')) else '  N/A'
        flag = r.get('text_helps', '?')
        print(f'{r["model"]:<12} {u:>10} {r["paper_uni"]:>11.4f} '
              f'{m:>12} {r["paper_multi"]:>13.4f} {dlt:>7} {flag:>6}')
    print('=' * 100)

    n_ok = df_all['model'].nunique()
    print(f'\nModels with results: {n_ok}/11')
    missing = set(ALL_MODELS) - set(df_all['model'].unique())
    if missing:
        print(f'Missing (no results yet): {sorted(missing)}')

In [ ]:
# ── Cell 18: Combined bar chart — all 11 models ──────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

if 'piv_all' not in dir() or piv_all.empty:
    print('⚠ Run Cell 17 first.')
else:
    # Display order: Regular → Large TS → Irregular
    MODEL_ORDER = [
        'DLinear', 'Informer', 'PatchTST', 'TimesNet', 'TimeMixer',
        'TimeLLM', 'TTM',
        'CRU', 'LatentODE', 'NeuralFlow', 'tPatchGNN',
    ]
    MODEL_CATEGORY = {
        'DLinear': 'Regular', 'Informer': 'Regular', 'PatchTST': 'Regular',
        'TimesNet': 'Regular', 'TimeMixer': 'Regular',
        'TimeLLM': 'Large TS', 'TTM': 'Large TS',
        'CRU': 'Irregular (ODE)', 'LatentODE': 'Irregular (ODE)',
        'NeuralFlow': 'Irregular (ODE)', 'tPatchGNN': 'Irregular (GNN)',
    }
    CAT_COLOR = {
        'Regular':         PALETTE['primary'],
        'Large TS':        PALETTE['secondary'],
        'Irregular (ODE)': PALETTE['accent'],
        'Irregular (GNN)': PALETTE['success'],
    }

    # Re-index to display order (keep models that have any data)
    plot_df = piv_all.set_index('model').reindex(
        [m for m in MODEL_ORDER if m in piv_all['model'].values]
    ).reset_index()

    models  = plot_df['model'].tolist()
    uni_v   = plot_df['mse_uni'].fillna(0).tolist()
    multi_v = plot_df['mse_multi'].fillna(0).tolist()
    changes = plot_df['mse_change_pct'].fillna(0).tolist()
    has_uni = plot_df['mse_uni'].notna().tolist()

    x = np.arange(len(models))
    w = 0.35

    fig, ax = plt.subplots(figsize=(15, 5.5))
    ax.bar(x - w/2, uni_v,   w, color=PALETTE['primary'],   label='Unimodal',           zorder=3)
    ax.bar(x + w/2, multi_v, w, color=PALETTE['secondary'], label='Multimodal (+ Text)', zorder=3)

    for i, (u, m, ch, ok) in enumerate(zip(uni_v, multi_v, changes, has_uni)):
        if not ok:
            ax.text(i, 0.04, 'N/A', ha='center', va='bottom', fontsize=8, color='gray')
            continue
        color = PALETTE['success'] if ch < 0 else PALETTE['accent']
        ax.text(i, max(u, m) + 0.014, f'{ch:+.1f}%',
                ha='center', va='bottom', fontsize=9, fontweight='bold', color=color)

    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=30, ha='right', fontsize=10)
    ax.set_ylabel('MSE (lower is better)', fontsize=11)
    ax.set_title('EPA-Air Forecasting: All 11 Models — Unimodal vs Multimodal (Combined)',
                 fontsize=13, fontweight='bold', color=PALETTE['neutral'], pad=12)
    ax.set_ylim(0, max(max(uni_v), max(multi_v)) * 1.16)
    ax.legend(loc='upper left', fontsize=10)

    # Colour x-tick labels by model category
    for lbl in ax.get_xticklabels():
        cat = MODEL_CATEGORY.get(lbl.get_text(), '')
        lbl.set_color(CAT_COLOR.get(cat, PALETTE['neutral']))

    # Category legend
    from matplotlib.patches import Patch
    cat_handles = [Patch(color=v, label=k) for k, v in CAT_COLOR.items()]
    ax.legend(handles=[
        plt.Rectangle((0,0),1,1, color=PALETTE['primary'],   label='Unimodal'),
        plt.Rectangle((0,0),1,1, color=PALETTE['secondary'], label='Multimodal (+Text)'),
    ] + cat_handles, fontsize=9, ncol=3, loc='upper center',
              bbox_to_anchor=(0.5, 1.0), frameon=False)

    plt.tight_layout()
    out = f'{PROJECT_DIR}/results/fig_ALL11_uni_vs_multi.png'
    plt.savefig(out, dpi=200, bbox_inches='tight', facecolor='white')
    print(f'✓ Saved to {out}')
    plt.show()

In [ ]:
# ── Cell 19: Paper vs Ours — all 11 models ───────────────────────────────────
if 'piv_all' not in dir() or piv_all.empty:
    print('⚠ Run Cell 17 first.')
else:
    plot_df = piv_all.set_index('model').reindex(
        [m for m in MODEL_ORDER if m in piv_all['model'].values]
    ).reset_index()

    n  = len(plot_df)
    x  = np.arange(n)
    w  = 0.2

    fig, ax = plt.subplots(figsize=(15, 5.5))

    ax.bar(x - 1.5*w, plot_df['paper_uni'],
           w, color=PALETTE['primary'],   alpha=0.35, label='Paper — Uni',
           zorder=3, edgecolor=PALETTE['primary'],   linewidth=0.8)
    ax.bar(x - 0.5*w, plot_df['mse_uni'].fillna(0),
           w, color=PALETTE['primary'],   alpha=1.0,  label='Ours — Uni',   zorder=3)
    ax.bar(x + 0.5*w, plot_df['paper_multi'],
           w, color=PALETTE['secondary'], alpha=0.35, label='Paper — Multi',
           zorder=3, edgecolor=PALETTE['secondary'], linewidth=0.8)
    ax.bar(x + 1.5*w, plot_df['mse_multi'].fillna(0),
           w, color=PALETTE['secondary'], alpha=1.0,  label='Ours — Multi', zorder=3)

    ax.set_xticks(x)
    ax.set_xticklabels(plot_df['model'].tolist(), rotation=30, ha='right', fontsize=10)
    ax.set_ylabel('MSE (lower is better)', fontsize=11)
    ax.set_title('All 11 Models: Paper vs Reproduction — EPA-Air (Table 11)',
                 fontsize=13, fontweight='bold', color=PALETTE['neutral'], pad=12)
    ax.legend(fontsize=9, ncol=2, loc='upper left')

    all_vals = pd.concat([
        plot_df['mse_uni'].fillna(0), plot_df['mse_multi'].fillna(0),
        plot_df['paper_uni'], plot_df['paper_multi'],
    ])
    ax.set_ylim(0, all_vals.max() * 1.18)

    plt.tight_layout()
    out = f'{PROJECT_DIR}/results/fig_ALL11_paper_vs_ours.png'
    plt.savefig(out, dpi=200, bbox_inches='tight', facecolor='white')
    print(f'✓ Saved to {out}')
    plt.show()

In [ ]:
# ── Cell 20: Full results table image (all 11 models) ────────────────────────
if 'piv_all' not in dir() or piv_all.empty:
    print('⚠ Run Cell 17 first.')
else:
    plot_df = piv_all.set_index('model').reindex(
        [m for m in MODEL_ORDER if m in piv_all['model'].values]
    ).reset_index()

    fig, ax = plt.subplots(figsize=(13, 5))
    ax.axis('off')

    col_labels = [
        'Model', 'Category',
        'MSE\n(Uni)', 'MSE\n(Multi)',
        'Paper Uni', 'Paper Multi',
        'Δ%', 'Text\nHelps?'
    ]

    cell_data   = []
    cell_colors = []
    for _, r in plot_df.iterrows():
        delta = r.get('mse_change_pct')
        helps = '✓' if pd.notna(delta) and delta < 0 else '✗'
        u = f'{r["mse_uni"]:.4f}'   if pd.notna(r.get('mse_uni'))   else 'N/A'
        m = f'{r["mse_multi"]:.4f}' if pd.notna(r.get('mse_multi')) else 'N/A'
        d = f'{delta:+.1f}%'        if pd.notna(delta)              else 'N/A'

        cat = MODEL_CATEGORY.get(r['model'], '')
        cell_data.append([
            r['model'], cat, u, m,
            f'{r["paper_uni"]:.4f}', f'{r["paper_multi"]:.4f}',
            d, helps,
        ])
        row_bg = '#e6f4ea' if helps == '✓' else '#fce8e6'
        cell_colors.append(['white'] * 6 + [row_bg, row_bg])

    tbl = ax.table(
        cellText=cell_data, colLabels=col_labels,
        cellColours=cell_colors,
        loc='center', cellLoc='center'
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9.5)
    tbl.scale(1, 1.55)

    for j in range(len(col_labels)):
        tbl[0, j].set_facecolor(PALETTE['primary'])
        tbl[0, j].set_text_props(color='white', fontweight='bold', fontsize=9)

    for i in range(len(cell_data)):
        tbl[i+1, 0].set_text_props(fontweight='bold')

    ax.set_title(
        'EPA-Air Reproduction Summary — All 11 Models (Paper Table 11)',
        fontsize=12, fontweight='bold', color=PALETTE['neutral'], pad=20
    )
    fig.text(0.5, 0.01,
             '* tPatchGNN run with patch_size=4 (default=24 crashes)  '
             '| TimeLLM run with patch_size=4  '
             '| ODE models (CRU, LatentODE, NeuralFlow) run with batch_size=4',
             ha='center', fontsize=8, color=PALETTE['neutral'], alpha=0.6)

    plt.tight_layout()
    out = f'{PROJECT_DIR}/results/fig_ALL11_results_table.png'
    plt.savefig(out, dpi=200, bbox_inches='tight', facecolor='white')
    print(f'✓ Saved to {out}')
    plt.show()